# Stage 5B — disposable touch/Pencil adapter

Use a Colab runtime already configured with LCProp 0.4.0 and ipympl. No installation or backend switch is performed here. Run no scientific calculation. This notebook does not touch your existing `app` or completed result.

Start with MODE = "synthetic". Only after adapter installation and gesture tests pass, change MODE to "beamplane" and rerun the cell for a **new disposable** packaged controller. Its Run button is disabled and only Configure is displayed. Do not use a completed scientific controller for this experiment.

This is an unqualified adapter. It relies on ipympl 0.12's two-canvas DOM order and synthetic mouse dispatch. If installation reports a framing/structure problem, **stop**. Do not traverse frames, inject code into widget managers or patch installed files. A successful scoped CSS experiment does not establish JavaScript access across Colab frames.

## Direct Colab setup

A newly opened notebook normally has a separate runtime. Follow the [immutable production 0.4.0 setup guide](https://github.com/mcroning/LCProp/blob/dfc426a26d288a540228d3ab24c990cf33105b0e/docs/notebook_package.md) in this notebook: manually install `lcprop[colab]==0.4.0`, restart the session, verify the version, enable Colab's custom widget manager and select `%matplotlib widget`. No setup command or installation is executed automatically by this diagnostic. Do not change the backend of an existing controller.

BeamPlane mode is gated: complete synthetic tests first, retain traces, and explicitly check the synthetic-validation checkbox. The gate also requires successful adapter preflight and an ordered Python press/motion/release sequence with valid coordinates. The checkbox attests to browser observations; Python events alone cannot prove finger/Pencil origin. If any synthetic check fails, leave it unchecked and stop. Download both traces and use **Remove adapter** before switching to beamplane mode.


In [ ]:
# Run this entire cell together: display and eval_js must use the SAME cell context.
MODE = "synthetic"  # Later: "beamplane"
import json, time, uuid, tempfile, dataclasses
from pathlib import Path
from importlib import metadata
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
from google.colab import output
assert "ipympl" in matplotlib.get_backend(), "Stop: initialize the qualified widget backend first"
assert metadata.version("lcprop") == "0.4.0"
output.enable_custom_widget_manager()
# Require prior synthetic evidence plus explicit hosted-observation acceptance.
if MODE == "beamplane":
    prior = globals().get("provenance5b", {})
    prior_events = globals().get("events5b", [])
    phase = 0
    for item in prior_events:
        if item["event"] == "button_press_event": phase = 1
        elif item["event"] == "figure_leave_event": phase = 0
        elif item["event"] == "motion_notify_event" and phase == 1 and item["xdata"] is not None and item["ydata"] is not None: phase = 2
        elif item["event"] == "button_release_event" and phase == 2: phase = 3; break
    if not (prior.get("mode") == "synthetic" and globals().get("result5b", {}).get("ok")
            and globals().get("approval5b") is not None and approval5b.value and phase == 3):
        raise RuntimeError("STOP: complete and explicitly accept synthetic browser validation before BeamPlane mode.")
if "diag5b" in globals():
    for cid in diag5b["ids"]:
        diag5b["canvas"].mpl_disconnect(cid)
    if diag5b["app"] is not None: diag5b["app"].close()
    else: plt.close(diag5b["fig"])
    diag5b["box"].close()
app5b = None
if MODE == "synthetic":
    with plt.ioff(): fig5b, ax5b = plt.subplots(figsize=(5, 4))
    ax5b.set(xlim=(0,100), ylim=(0,100), title="Synthetic pointer diagnostic — no science")
    ax5b.grid(True)
    canvas5b = fig5b.canvas
    content5b = canvas5b
elif MODE == "beamplane":
    from lcprop.notebook import InteractiveLC
    with plt.ioff(): app5b = InteractiveLC()
    app5b.run_button.disabled = True
    app5b.live_preview.value = False  # Marker-only proposals; numeric commit on valid release.
    editor5b = app5b._position_editor  # Documented diagnostic-only private coupling.
    assert editor5b is not None
    canvas5b = editor5b.canvas
    fig5b = canvas5b.figure
    content5b = app5b.configure.children[0]
else: raise ValueError("Unknown MODE")
canvas5b.pan_zoom_throttle = 0  # Set before displaying the new view; no delayed 33-ms trailing move.
canvas5b.capture_scroll = False
canvas5b.resizable = False
canvas5b.toolbar_visible = False
scope5b = "lcprop-touch-5b-" + uuid.uuid4().hex
canvas5b.add_class(scope5b)
events5b = []
trial5b = W.Text(description="Trial", value=MODE + "-finger-1")
save5b = W.Button(description="Save Python evidence")
status5b = W.HTML(value="Waiting for adapter preflight")
provenance5b = {"versions": {n:metadata.version(n) for n in ("lcprop","matplotlib","ipympl","ipywidgets","IPython")},
    "backend":matplotlib.get_backend(), "mode":MODE,
    "requested_frontend":str(canvas5b._model_module_version), "actual_frontend":"UNVERIFIED"}
def record5b(e):
    if len(events5b)>=2000: return
    entry={"seq":len(events5b), "time":time.monotonic(), "trial":trial5b.value,
        "event":e.name,"x":e.x,"y":e.y,"xdata":e.xdata,"ydata":e.ydata,"button":str(getattr(e,"button",None))}
    if app5b is not None:
        entry["request"] = dataclasses.asdict(app5b.request())
        entry["completed_is_none"] = app5b.completed is None
    events5b.append(entry)
ids5b = [canvas5b.mpl_connect(n,record5b) for n in ("button_press_event","motion_notify_event","button_release_event","figure_leave_event")]
def save_python5b(_):
    folder=Path(tempfile.mkdtemp(prefix="lcprop-touch-5b-"))
    p=folder/"python-evidence.json"
    p.write_text(json.dumps({"provenance":provenance5b,"events":events5b,"scientific_execution":False},indent=2))
    status5b.value="Download through Colab Files: " + str(p)
save5b.on_click(save_python5b)
approval5b = W.Checkbox(value=False, disabled=MODE != "synthetic",
    description="Synthetic finger/Pencil, cancellation, scrolling and available mouse tests passed; traces retained",
    indent=False, layout=W.Layout(width="auto"))
box5b=W.VBox([W.HBox([trial5b,save5b]),status5b,approval5b,content5b])
diag5b={"app":app5b,"canvas":canvas5b,"fig":fig5b,"ids":ids5b,"box":box5b}
display(box5b)
ADAPTER_SOURCE = "/* Disposable diagnostic. No ipympl model/view-manager access, package writes or Run calls. */\nfunction installTouchAdapter(scope, doc = document) {\n  if (!/^lcprop-touch-5b-[a-f0-9]+$/.test(scope)) throw Error('Invalid diagnostic scope');\n  const roots = doc.querySelectorAll('.' + scope);\n  if (roots.length !== 1) return {ok:false, reason:'Scoped widget not in this output document. STOP; no frame traversal.'};\n  const root = roots[0];\n  const canvases = [...root.querySelectorAll('canvas')];\n  if (canvases.length !== 2) return {ok:false, reason:'Expected two ipympl canvases. STOP; frontend structure unqualified.'};\n  // Version-specific DOM contract: ipympl 0.12 top interaction canvas follows image canvas.\n  const target = canvases[1], win = doc.defaultView;\n  if (!win.PointerEvent) return {ok:false, reason:'Pointer Events unavailable'};\n  if (root.__lcpropTouch5b) return {ok:false, reason:'Adapter already installed; remove it before reinstalling'};\n  const trace = [], listeners = [], synthetic = new WeakSet();\n  let active = null, blockedUntil = 0, nextStart = 0, dropped = 0;\n  const now = () => win.performance.now();\n  function log(kind, e, extra = {}) {\n    if (trace.length >= 2000) { dropped++; return; }\n    trace.push({seq:trace.length, ms:now(), kind, pointerId:e?.pointerId,\n      pointerType:e?.pointerType, clientX:e?.clientX, clientY:e?.clientY, ...extra});\n  }\n  function inside(e) {\n    const r = target.getBoundingClientRect();\n    return Number.isFinite(e.clientX) && Number.isFinite(e.clientY) &&\n      e.clientX >= r.left && e.clientX < r.right && e.clientY >= r.top && e.clientY < r.bottom;\n  }\n  function send(type, e, buttons) {\n    // Preserve CSS client coordinates: ipympl applies its own bounding-box/pixel ratio conversion.\n    const m = new win.MouseEvent(type, {bubbles:true, cancelable:true, view:win,\n      clientX:e.clientX, clientY:e.clientY, button:0, buttons,\n      ctrlKey:false, shiftKey:false, altKey:false, metaKey:false});\n    synthetic.add(m); log('emit:' + type, e, {buttons}); target.dispatchEvent(m);\n  }\n  function dropCapture(id) {\n    if (target.hasPointerCapture(id)) target.releasePointerCapture(id);\n  }\n  function cancel(reason) {\n    if (!active) return;\n    const old = active; active = null; blockedUntil = now()+800; nextStart = now()+100;\n    log('cancel', old.last, {reason});\n    // Existing BeamPlane figure_leave handler discards provisional geometry; never synthesize mouseup.\n    send('mouseleave', old.last, 0); dropCapture(old.id);\n  }\n  function stop(e) { e.preventDefault(); e.stopImmediatePropagation(); }\n  function down(e) {\n    log('pointerdown',e);\n    if (e.pointerType === 'mouse') {\n      cancel('mouse-interruption'); blockedUntil = 0; return; // Leave genuine desktop path untouched.\n    }\n    if (!['touch','pen'].includes(e.pointerType)) return;\n    stop(e);\n    if (active) { cancel('second-contact'); return; }\n    if (!e.isPrimary || e.button !== 0 || now() < nextStart || !inside(e)) return;\n    try { target.setPointerCapture(e.pointerId); }\n    catch (_) { log('capture-failed', e); return; }\n    active = {id:e.pointerId, last:e}; blockedUntil=now()+800;\n    send('mousedown', e, 1);\n  }\n  function move(e) {\n    if (!active || active.id !== e.pointerId) return;\n    stop(e); active.last=e;\n    if (!inside(e) || !(e.buttons & 1)) { cancel('outside-or-contact-ended'); return; }\n    send('mousemove', e, 1);\n  }\n  function up(e) {\n    if (!active || active.id !== e.pointerId) return;\n    stop(e); active.last=e;\n    if (!inside(e)) { cancel('release-outside'); return; }\n    const id=active.id; active=null; blockedUntil=now()+800; nextStart=now()+100;\n    send('mouseup', e, 0); dropCapture(id);\n  }\n  function interrupted(e) { if (active && active.id===e.pointerId) cancel(e.type); }\n  function compat(e) {\n    if (synthetic.has(e)) return;\n    if (active || now()<blockedUntil || e.sourceCapabilities?.firesTouchEvents) {\n      stop(e); log('suppressed:'+e.type,e);\n    }\n  }\n  function on(obj,type,fn,options) { obj.addEventListener(type,fn,options); listeners.push([obj,type,fn,options]); }\n  const oldTouch=target.style.touchAction;\n  target.style.touchAction='none';\n  on(target,'pointerdown',down,{capture:true,passive:false});\n  on(target,'pointermove',move,{capture:true,passive:false});\n  on(target,'pointerup',up,{capture:true,passive:false});\n  on(target,'pointercancel',interrupted,true);\n  on(target,'lostpointercapture',interrupted,true);\n  on(target,'pointerleave',e=>{if(active && e.pointerId===active.id) cancel('pointerleave');},true);\n  // Do not suppress real figure-leave: it remains an independent cancellation route.\n  for(const type of ['mousedown','mousemove','mouseup','click','dblclick']) on(target,type,compat,true);\n  on(doc,'pointerdown',e=>{if(active && e.pointerId!==active.id) cancel('other-contact');},true);\n  on(win,'blur',()=>cancel('window-blur'));\n  on(doc,'visibilitychange',()=>{if(doc.hidden) cancel('document-hidden');});\n  const api={trace, snapshot:()=>({trace:[...trace],dropped,active:!!active}),\n    cancel:()=>cancel('manual'),\n    remove:()=>{cancel('remove'); for(const x of listeners)x[0].removeEventListener(x[1],x[2],x[3]);\n      target.style.touchAction=oldTouch; delete root.__lcpropTouch5b;}};\n  root.__lcpropTouch5b=api;\n  log('installed',null,{canvasCount:canvases.length,touchAction:win.getComputedStyle(target).touchAction});\n  return {ok:true,scope,canvasCount:canvases.length,touchAction:win.getComputedStyle(target).touchAction,\n    note:'Diagnostic DOM contract only. Browser qualification required.'};\n}\n"
script5b = ADAPTER_SOURCE + "\n" + """
(async () => {
 const scope = SCOPE;
 // Wait briefly for widget rendering in this document; never traverse iframe or parent documents.
 for(let i=0;i<25 && !document.querySelector('.'+scope);i++) await new Promise(r=>setTimeout(r,200));
 const result=installTouchAdapter(scope);
 if(!result.ok) return result;
 const root=document.querySelector('.'+scope), api=root.__lcpropTouch5b;
 const controls=document.createElement('div');
 const label=document.createElement('input');label.placeholder='Browser trial label (same as Python)';
 const mark=document.createElement('button');mark.textContent='Mark trial';
 mark.onclick=()=>api.trace.push({kind:'trial',label:label.value,ms:performance.now()});
 const download=document.createElement('button');download.textContent='Download browser trace';
 download.onclick=()=>{const data={...api.snapshot(),userAgent:navigator.userAgent,scope};
   const url=URL.createObjectURL(new Blob([JSON.stringify(data,null,2)],{type:'application/json'}));
   const a=document.createElement('a');a.href=url;a.download='browser-evidence.json';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);};
 const remove=document.createElement('button');remove.textContent='Remove adapter';
 remove.onclick=()=>{api.remove();controls.remove();};
 controls.append(label,mark,download,remove);root.appendChild(controls);
 return result;
})()
""".replace("SCOPE", json.dumps(scope5b))
try:
    result5b = output.eval_js(script5b, timeout_sec=15)
except Exception as error:
    raise RuntimeError("STOP: Colab cannot execute the same-document adapter preflight. Retain this error; no frame workaround.") from error
print(result5b)
if not result5b or not result5b.get("ok"):
    raise RuntimeError("STOP: adapter unavailable in widget document. Retain result; no invasive fallback.")
status5b.value="Installed experimentally. Record browser and Python trial labels; no Run."


## Qualification — no science

Retain installation output, actual device/iPadOS/browser versions, and both trace files. Browser trace records pointer types, cancellations and compatibility suppression; Python trace records Matplotlib coordinates and request state. Do not call a missing frontend version verified.

1. Synthetic canvas: finger, Pencil, mouse separately; complete press/motion/release, then interrupted/outside/multi-contact gestures. Mouse tests require actual hardware; mark unavailable tests untested.
2. Verify scrolling outside the canvas; verify adapter removal restores prior behavior. There must be one effective press/release per successful gesture and no release on cancellation. Synthetic mouse events intentionally have `isTrusted=false`.
3. Only after step 1 passes, use fresh `beamplane` mode. Select Active Beam and Action: Move, drag its center; select Tilt and drag its ray endpoint. Test Beam 2 as well. Verify numeric controls update on release and numeric edits redraw the canvas. This experiment does not map physical stylus tilt to optical tilt.
4. Cancel via leaving canvas, second contact, page interruption, or loss of capture. No partial request change may survive. Retain before/after numeric values and Python request trace. Run is disabled; completed must stay absent. This does not qualify preservation of a real completed scientific snapshot.
5. Compare desktop mouse with adapter removed and installed. No double commits. Download browser trace before removing the adapter or rerunning the cell, and save Python evidence. Do not infer touch success from local mocks.

**Stop** if the widget is inaccessible from the cell document, its structure differs, synthetic events are ignored, trailing/cancellation events cause commits, or a later gesture consumes stale coordinates. No iframe traversal, private widget-manager injection or installed-file patches are authorized.
